# FactIndic: Roman-script Hinglish summarization + factuality pilot

**This is the corrected experiment.** Inputs must be Roman-script Hindi-English (Hindi transliterated into Latin letters, mixed with English), paired with the official English summary. The previous Devanagari-based pilot is not used here.

This notebook expects the authorized **GupShup h2e `test.source` and `test.target`** files. The teacher has confirmed that any dataset may be used. GupShup conversations were translated from SAMSum, so report that domain limitation clearly. The dataset release is gated and says not to redistribute; keep source files, generated summaries, review sheets, and keys in private Drive only. Never commit them to the public GitHub repository.

Study design: 30 fixed test conversations × 2 LLMs × 2 prompts = 120 summaries. This is an exploratory, single-annotator pilot. It can support descriptive results, not broad benchmark claims. Primary factuality evidence comes from a blinded human source-grounded review; ROUGE and any automated scores do not substitute for that review.

Model candidates are similar in size and multilingual intent: Qwen3-4B and Gemma 3 4B IT. Qwen3 advertises support for 100+ languages; Gemma 3 for 140+ languages and summarization. Gemma access requires accepting Google's model terms on Hugging Face. The notebook will prompt for a private HF token. Model cards: [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) · [Gemma 3 4B IT](https://huggingface.co/google/gemma-3-4b-it).

**Before generation:** obtain GupShup access, accept Gemma terms if using that comparison, and inspect the sampled conversations for Roman-script Hindi-English mixing. Teacher dataset approval is recorded in Cell 10; do not unlock generation until the sample language check is complete.

## 1. Install the small inference/evaluation stack

Choose a **GPU** runtime in Colab. The notebook checks CUDA before model loading. Model inference stays in Colab; this does not use your Mac's CPU/GPU.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece rouge-score

In [ ]:
import csv, gc, hashlib, json, random, re, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
from getpass import getpass

import torch
from google.colab import drive, files
from huggingface_hub import login
from rouge_score import rouge_scorer
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > GPU, then rerun from the top."
print("GPU:", torch.cuda.get_device_name(0))

# Private persistent workspace; do not share this Drive folder.
drive.mount('/content/drive')
PRIVATE_ROOT = Path('/content/drive/MyDrive/FactIndic_Private')
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Upload the authorized test split and check alignment

Upload **only** the GupShup h2e official test pair: `test.source` and `test.target`. The source file contains one conversation per line; the target has the aligned English summary on the same line. Do not upload train files for this evaluation.

In [ ]:
uploaded = files.upload()
source_name = next((n for n in uploaded if n.endswith('.source')), None)
target_name = next((n for n in uploaded if n.endswith('.target')), None)
if not source_name or not target_name:
    raise ValueError("Upload the official h2e test.source and test.target files together.")

data_dir = PRIVATE_ROOT / 'gupshup_h2e'
data_dir.mkdir(parents=True, exist_ok=True)
source_path = data_dir / 'test.source'
target_path = data_dir / 'test.target'
source_path.write_bytes(uploaded[source_name])
target_path.write_bytes(uploaded[target_name])
del uploaded
gc.collect()

sources = source_path.read_text(encoding='utf-8-sig').splitlines()
references = target_path.read_text(encoding='utf-8-sig').splitlines()
if len(sources) != len(references):
    raise ValueError(f"Alignment failed: {len(sources)} source rows vs {len(references)} target rows.")
if not sources:
    raise ValueError("The uploaded test split is empty.")

DEV = re.compile(r'[\u0900-\u097F]')
LATIN = re.compile(r'[A-Za-z]')
source_dev = sum(bool(DEV.search(x)) for x in sources)
source_latin = sum(bool(LATIN.search(x)) for x in sources)
blank_sources = sum(not x.strip() for x in sources)
blank_refs = sum(not x.strip() for x in references)
print(f"Aligned pairs: {len(sources)}")
print(f"Source rows with Latin letters: {source_latin}/{len(sources)}")
print(f"Source rows containing Devanagari: {source_dev}/{len(sources)}")
print(f"Blank source/reference rows: {blank_sources}/{blank_refs}")
if blank_sources or blank_refs:
    raise ValueError("Blank row(s) detected. Investigate alignment before continuing.")

# Stable sample and input fingerprints for reproducibility.
SEED = 42
N_SAMPLE = 30
if len(sources) < N_SAMPLE:
    raise ValueError(f"Expected at least {N_SAMPLE} test pairs; found {len(sources)}.")
rng = random.Random(SEED)
selected = sorted(rng.sample(range(len(sources)), N_SAMPLE))
sample = [{'source_id': f'test_line_{i+1:06d}', 'source': sources[i], 'reference': references[i]} for i in selected]
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
print("Sample size:", len(sample), "| seed:", SEED)
print("Source SHA-256:", sha(source_path))
print("Target SHA-256:", sha(target_path))


## 3. Inspect the sample and confirm the dataset fits

Check that the sampled sources are actually Hindi-English code-mixed in Latin script and that the official reference summaries align with the conversations. The teacher has approved using any dataset; retain the translated-conversation limitation in reporting. This is a real data-validity checkpoint, not a model result.

In [ ]:
from IPython.display import display
import pandas as pd

preview_rng = random.Random(SEED + 1)
preview_ids = sorted(preview_rng.sample(range(len(sample)), min(8, len(sample))))
display(pd.DataFrame([{'source_id': sample[i]['source_id'], 'source': sample[i]['source'], 'English reference': sample[i]['reference']} for i in preview_ids]))

print("The next configuration cell keeps generation locked until you record both approvals.")


## 4. Configure paired model and prompt conditions

Use the same 30 source IDs for every condition. Each model receives a 768-token source budget using its own tokenizer; truncation is recorded per output. Greedy decoding and the output cap stay fixed. Qwen3 non-thinking mode is requested to keep the task focused on summarization.

In [ ]:
# The teacher approved using any dataset. After inspecting the sample yourself,
# change SAMPLE_LANGUAGE_CHECKED to True before starting generation.
SAMPLE_LANGUAGE_CHECKED = False
TEACHER_ACCEPTED_DATASET = True
assert SAMPLE_LANGUAGE_CHECKED, "Inspect the sample first, then change SAMPLE_LANGUAGE_CHECKED to True."
assert TEACHER_ACCEPTED_DATASET, "Teacher dataset approval is required before generation."

MODEL_IDS = {
    'qwen3_4b': 'Qwen/Qwen3-4B',
    'gemma3_4b_it': 'google/gemma-3-4b-it',
}
PROMPTS = {
    'standard': (
        "Summarize this Hindi-English conversation in clear, concise English. "
        "Preserve the central events, participants, and important details. "
        "Use only information stated in the conversation."
    ),
    'factuality_aware': (
        "Write a concise English summary using only facts explicitly stated in this conversation. "
        "Do not infer causes, intentions, dates, quantities, identities, or outcomes. "
        "Preserve who did what; if a detail is unclear, omit it rather than guess."
    ),
}
MAX_SOURCE_TOKENS = 768
MAX_NEW_TOKENS = 128
DTYPE = torch.float16

# Gemma is gated. Accept its terms on Hugging Face first, then paste a read token here.
hf_token = getpass('Hugging Face token with Gemma access (input is hidden): ')
if not hf_token.strip():
    raise ValueError('A Hugging Face token is required for the Gemma model.')
login(token=hf_token, add_to_git_credential=False)
del hf_token

run_fingerprint = hashlib.sha256(json.dumps({'source_sha256':sha(source_path),'target_sha256':sha(target_path),'seed':SEED,'n_sample':N_SAMPLE,'models':MODEL_IDS,'prompts':PROMPTS},sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = f'romanized_hinglish_pilot_{run_fingerprint}'
RUN_DIR = PRIVATE_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
GENERATIONS_PATH = RUN_DIR / 'generations.jsonl'
print('Private run directory:', RUN_DIR)
print('Conditions:', len(MODEL_IDS), 'models x', len(PROMPTS), 'prompts x', len(sample), 'sources')


## 5. Run the paired generation

Models load sequentially to fit a Colab T4 when available. Each completed output is checkpointed to private Drive so an interrupted run can resume. Re-running skips existing `(source_id, model_key, prompt_arm)` outputs. Do not share the private run directory.

In [ ]:
def load_runner(model_key, model_id):
    if model_key == 'gemma3_4b_it':
        from transformers import AutoProcessor, Gemma3ForConditionalGeneration
        processor = AutoProcessor.from_pretrained(model_id, token=True)
        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=DTYPE, device_map='auto', token=True
        ).eval()
        return model, processor, processor.tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map='auto', token=True
    ).eval()
    return model, tokenizer, tokenizer

def source_for_model(tokenizer, source):
    ids = tokenizer.encode(source, add_special_tokens=False)
    truncated = len(ids) > MAX_SOURCE_TOKENS
    clipped = tokenizer.decode(ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
    return clipped, len(ids), truncated

def make_inputs(model_key, processor_or_tokenizer, prompt_text):
    if model_key == 'gemma3_4b_it':
        messages = [{'role':'user','content':[{'type':'text','text':prompt_text}]}]
        return processor_or_tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt'
        )
    messages = [{'role':'user','content':prompt_text}]
    return processor_or_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt', enable_thinking=False
    )

def decode_generated(model_key, processor_or_tokenizer, generated_ids):
    if model_key == 'gemma3_4b_it':
        return processor_or_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return processor_or_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def existing_keys(path):
    found = {}
    if path.exists():
        with path.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    found[(row['source_id'], row['model_key'], row['prompt_arm'])] = row
    return found

results_by_key = existing_keys(GENERATIONS_PATH)
total = len(sample) * len(MODEL_IDS) * len(PROMPTS)
completed = len(results_by_key)
print(f'{completed}/{total} outputs already checkpointed')
for model_key, model_id in MODEL_IDS.items():
    needed = [(item, arm, instruction) for item in sample for arm, instruction in PROMPTS.items()
              if (item['source_id'], model_key, arm) not in results_by_key]
    if not needed:
        print(f'{model_key}: all outputs already checkpointed; skipping model load.')
        continue
    print('Loading', model_id)
    model, processor_obj, tokenizer = load_runner(model_key, model_id)
    for item, arm, instruction in needed:
        clipped_source, input_source_tokens, truncated = source_for_model(tokenizer, item['source'])
        prompt_text = f"{instruction}\n\nConversation:\n{clipped_source}\n\nEnglish summary:"
        inputs = make_inputs(model_key, processor_obj, prompt_text).to(model.device)
        input_length = int(inputs['input_ids'].shape[-1])
        start = time.time()
        with torch.inference_mode():
            output = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                num_beams=1, pad_token_id=(tokenizer.pad_token_id or tokenizer.eos_token_id)
            )
        generated_ids = output[0][input_length:]
        prediction = decode_generated(model_key, processor_obj, generated_ids)
        row = {
            'source_id': item['source_id'], 'source': item['source'], 'reference': item['reference'],
            'model_key': model_key, 'model_id': model_id, 'prompt_arm': arm,
            'prediction': prediction, 'source_input_tokens': input_source_tokens,
            'source_truncated': bool(truncated), 'input_tokens_with_prompt': input_length,
            'generated_tokens': int(len(generated_ids)),
            'hit_new_token_cap': int(len(generated_ids)) >= MAX_NEW_TOKENS,
            'inference_seconds': round(time.time() - start, 3),
        }
        results_by_key[(item['source_id'], model_key, arm)] = row
        with GENERATIONS_PATH.open('a', encoding='utf-8') as f:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            f.flush()
        completed += 1
        if completed % 10 == 0 or completed == total:
            print(f'{completed}/{total} outputs checkpointed')
    del model, processor_obj, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

rows = list(results_by_key.values())
expected = len(sample) * len(MODEL_IDS) * len(PROMPTS)
assert len(rows) == expected, f'Expected {expected} generations, found {len(rows)}.'
print('Completed paired generation:', len(rows))


## 6. Quality gate, reference metrics, and blind review pack

Generation-quality checks can block annotation when outputs are empty, hit the new-token cap, or contain replacement characters. Missing final punctuation is reported for inspection but does not automatically fail a summary. Reference-based ROUGE is reported separately from factuality. The blind review sheet hides the model and prompt arm; a private key stays in Drive and is excluded from the annotation ZIP.

In [ ]:
from collections import Counter, defaultdict

cap_hits = sum(bool(r['hit_new_token_cap']) for r in rows)
empty_rows = sum(not r['prediction'].strip() for r in rows)
replacement_rows = sum('�' in r['prediction'] for r in rows)
terminal_marks = ('.','!','?','…','”','"')
no_terminal = sum(not r['prediction'].rstrip().endswith(terminal_marks) for r in rows)
status = 'diagnostic_only' if (cap_hits or empty_rows or replacement_rows) else 'annotation_candidate'
quality = {
    'status':status,'rows':len(rows),'cap_hits':cap_hits,'empty_rows':empty_rows,
    'replacement_char_rows':replacement_rows,'without_terminal_punctuation':no_terminal,
    'factuality_is_not_automatically_verified':True
}
(RUN_DIR/'quality_audit.json').write_text(json.dumps(quality,indent=2),encoding='utf-8')
run_metadata = {'run_id':RUN_ID,'task':'GupShup h2e','status':status,'n_sources':len(sample),'n_generations':len(rows),'seed':SEED,'models':MODEL_IDS,'prompt_arms':PROMPTS,'max_source_tokens_per_model':MAX_SOURCE_TOKENS,'max_new_tokens':MAX_NEW_TOKENS,'decoding':'greedy, do_sample=False, num_beams=1','input_sha256':{'test.source':sha(source_path),'test.target':sha(target_path)},'selected_source_line_ids':[r['source_id'] for r in sample],'transformers_version':transformers.__version__,'torch_version':torch.__version__,'gpu':torch.cuda.get_device_name(0),'human_annotation_status':'not_started','automated_metrics_are_not_factuality_labels':True}
(RUN_DIR/'run_metadata.json').write_text(json.dumps(run_metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print(quality)

# ROUGE is reference overlap only; source-grounded factuality is annotated separately.
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
for r in rows:
    score = scorer.score(r['reference'], r['prediction'])
    r['rouge1_f1'] = round(score['rouge1'].fmeasure, 6)
    r['rouge2_f1'] = round(score['rouge2'].fmeasure, 6)
    r['rougeL_f1'] = round(score['rougeL'].fmeasure, 6)

metric_groups = defaultdict(list)
for r in rows:
    metric_groups[(r['model_key'],r['prompt_arm'])].append(r)
metric_rows=[]
for (model_key,arm), group in sorted(metric_groups.items()):
    metric_rows.append({'model_key':model_key,'prompt_arm':arm,'n':len(group),
        **{k:round(sum(x[k] for x in group)/len(group),4) for k in ['rouge1_f1','rouge2_f1','rougeL_f1']},
        'mean_inference_seconds':round(sum(x['inference_seconds'] for x in group)/len(group),3),
        'source_truncated_rows':sum(x['source_truncated'] for x in group)})
with (RUN_DIR/'rouge_summary.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(metric_rows[0]));w.writeheader();w.writerows(metric_rows)
print('ROUGE summary saved. These scores do not establish factuality.')

def bootstrap_mean_diff(a, b, seed=42, iterations=2000):
    rng=random.Random(seed)
    diffs=[x-y for x,y in zip(a,b)]
    n=len(diffs)
    draws=[sum(rng.choice(diffs) for _ in range(n))/n for _ in range(iterations)]
    return sum(diffs)/n, sorted(draws)[int(.025*iterations)], sorted(draws)[int(.975*iterations)]

pair_metrics=[]
by_key={(r['source_id'],r['model_key'],r['prompt_arm']):r for r in rows}
for model_key in MODEL_IDS:
    for metric in ['rouge1_f1','rouge2_f1','rougeL_f1']:
        left=[by_key[(s['source_id'],model_key,'factuality_aware')][metric] for s in sample]
        right=[by_key[(s['source_id'],model_key,'standard')][metric] for s in sample]
        mean,lo,hi=bootstrap_mean_diff(left,right,seed=SEED)
        pair_metrics.append({'comparison':f'{model_key}: factuality_aware - standard','metric':metric,'n_pairs':len(sample),'mean_difference':round(mean,4),'ci95_low':round(lo,4),'ci95_high':round(hi,4)})
for arm in PROMPTS:
    for metric in ['rouge1_f1','rouge2_f1','rougeL_f1']:
        left=[by_key[(s['source_id'],'gemma3_4b_it',arm)][metric] for s in sample]
        right=[by_key[(s['source_id'],'qwen3_4b',arm)][metric] for s in sample]
        mean,lo,hi=bootstrap_mean_diff(left,right,seed=SEED)
        pair_metrics.append({'comparison':f'gemma3_4b_it - qwen3_4b: {arm}','metric':metric,'n_pairs':len(sample),'mean_difference':round(mean,4),'ci95_low':round(lo,4),'ci95_high':round(hi,4)})
with (RUN_DIR/'paired_rouge_bootstrap.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(pair_metrics[0]));w.writeheader();w.writerows(pair_metrics)

# Save complete generation rows and private unblinding key.
fields=list(rows[0])
with (RUN_DIR/'generations_with_rouge.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
blind_rows=[]; key_rows=[]
for r in rows:
    blind_rows.append({'source_id':r['source_id'],'model_key':r['model_key'],'model_id':r['model_id'],'prompt_arm':r['prompt_arm'],
        'source':r['source'],'prediction':r['prediction'],'human_factuality':'','error_type':'','evidence_or_error_span':'','notes':''})
random.Random(SEED+9).shuffle(blind_rows)
for i,r in enumerate(blind_rows,1):
    blind_id=f'review_{i:04d}'
    r['blind_id']=blind_id
    key_rows.append({'blind_id':blind_id,'source_id':r['source_id'],'model_key':r['model_key'],'model_id':r['model_id'],'prompt_arm':r['prompt_arm']})
review_fields=['blind_id','source','prediction','human_factuality','error_type','evidence_or_error_span','notes']
# The review sheet contains no source IDs, model names, or prompt-arm labels.
with (RUN_DIR/'human_review_blinded.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=review_fields);w.writeheader();w.writerows([{k:r[k] for k in review_fields} for r in blind_rows])
with (RUN_DIR/'private_unblinding_key.csv').open('w',encoding='utf-8-sig',newline='') as f:
    keys=['blind_id','source_id','model_key','model_id','prompt_arm']
    w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(key_rows)

protocol = """FactIndic source-grounded factuality review

For each summary, compare it with the source conversation. Do not use the reference summary to judge factuality.

human_factuality labels:
- supported: no material claim conflicts with or is absent from the source.
- minor_error: a small factual error or omission; central meaning remains accurate.
- major_error: a material unsupported or incorrect claim changes the central meaning.
- cannot_determine: the source does not resolve the claim.

Record an error type where applicable: unsupported_addition, entity, number/date, relation, attribution, polarity, omission, or other. Include the shortest useful evidence/error span. One annotator is available; report this limitation and do not claim inter-annotator agreement.
"""
(RUN_DIR/'review_protocol.txt').write_text(protocol,encoding='utf-8')

annotation_zip=RUN_DIR/'factindic_blinded_human_review_pack.zip'
with zipfile.ZipFile(annotation_zip,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(RUN_DIR/'human_review_blinded.csv',arcname='human_review_blinded.csv')
    z.write(RUN_DIR/'review_protocol.txt',arcname='review_protocol.txt')
print('Annotation status:',status)
if status == 'annotation_candidate':
    print('Manual source review required for all',len(rows),'summaries before making factuality claims.')
else:
    print('Diagnostic only. Do not annotate this run; resolve generation issues and rerun.')
print('Private review pack:',annotation_zip)
files.download(str(annotation_zip))


## 7. After you complete the blind human review

Fill every `human_factuality` cell in the CSV with `supported`, `minor_error`, `major_error`, or `cannot_determine`. Record evidence spans. Upload the completed CSV below. Only after all labels are complete will the notebook join the private key and compute descriptive condition totals. The reviewer must not fill labels from model outputs, NLI scores, or the reference summaries.

In [ ]:
filled_upload = files.upload()
review_name = next((n for n in filled_upload if n.lower().endswith('.csv')), None)
if not review_name:
    raise ValueError('Upload the completed human_review_blinded.csv file.')
review_path = RUN_DIR / 'human_review_blinded_filled.csv'
review_path.write_bytes(filled_upload[review_name])

with review_path.open(encoding='utf-8-sig',newline='') as f:
    reviewed=list(csv.DictReader(f))
required={'blind_id','human_factuality'}
if not reviewed or not required.issubset(reviewed[0]):
    raise ValueError(f'Missing required review columns: {sorted(required-set(reviewed[0] if reviewed else []))}')
allowed={'supported','minor_error','major_error','cannot_determine'}
labels=[r['human_factuality'].strip().lower() for r in reviewed]
if any(x not in allowed for x in labels):
    raise ValueError('Every row needs one permitted human_factuality label before analysis.')
if len({r['blind_id'] for r in reviewed}) != len(reviewed):
    raise ValueError('Duplicate blind_id values found.')

key_path=RUN_DIR/'private_unblinding_key.csv'
with key_path.open(encoding='utf-8-sig',newline='') as f:
    key={r['blind_id']:r for r in csv.DictReader(f)}
if set(r['blind_id'] for r in reviewed) != set(key):
    raise ValueError('Review rows do not match the private unblinding key.')

joined=[]
for r in reviewed:
    meta=key[r['blind_id']]
    joined.append({**r,**meta,'human_factuality':r['human_factuality'].strip().lower(),
                   'is_error':r['human_factuality'].strip().lower() in {'minor_error','major_error'}})
summary=[]
for model_key in MODEL_IDS:
    for arm in PROMPTS:
        group=[r for r in joined if r['model_key']==model_key and r['prompt_arm']==arm]
        counts=Counter(r['human_factuality'] for r in group)
        determinate=[r for r in group if r['human_factuality']!='cannot_determine']
        errors=sum(r['is_error'] for r in determinate)
        summary.append({'model_key':model_key,'prompt_arm':arm,'n':len(group),
            **{label:counts[label] for label in sorted(allowed)},
            'determinate_n':len(determinate),
            'minor_or_major_error_rate_among_determinate':round(errors/len(determinate),4) if determinate else None,
            'cannot_determine_rate':round(counts['cannot_determine']/len(group),4) if group else None,
            'annotator_count':1})
with (RUN_DIR/'human_factuality_summary.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(summary[0]));w.writeheader();w.writerows(summary)
with (RUN_DIR/'unblinded_human_review_private.csv').open('w',encoding='utf-8-sig',newline='') as f:
    fields=list(joined[0]);w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(joined)
print('Human factuality totals (single annotator; descriptive only):')
for row in summary: print(row)

human_lookup={(r['source_id'],r['model_key'],r['prompt_arm']):r for r in joined}
human_pairs=[]
def paired_error_differences(comparator_a, comparator_b):
    diffs=[];excluded=0
    for s in sample:
        a=human_lookup[comparator_a(s)]
        b=human_lookup[comparator_b(s)]
        if a['human_factuality']=='cannot_determine' or b['human_factuality']=='cannot_determine':
            excluded+=1
        else:
            diffs.append(int(a['is_error'])-int(b['is_error']))
    if not diffs: return None,None,None,0,excluded
    mean,lo,hi=bootstrap_mean_diff([float(x) for x in diffs],[0.0]*len(diffs),seed=SEED)
    return mean,lo,hi,len(diffs),excluded

for model_key in MODEL_IDS:
    mean,lo,hi,n_pairs,excluded=paired_error_differences(
        lambda s:(s['source_id'],model_key,'factuality_aware'),
        lambda s:(s['source_id'],model_key,'standard'))
    human_pairs.append({'comparison':f'{model_key}: factuality_aware - standard','n_determinate_pairs':n_pairs,'pairs_excluded_for_cannot_determine':excluded,'error_rate_difference':round(mean,4) if mean is not None else None,'ci95_low':round(lo,4) if lo is not None else None,'ci95_high':round(hi,4) if hi is not None else None,'annotator_count':1})
for arm in PROMPTS:
    mean,lo,hi,n_pairs,excluded=paired_error_differences(
        lambda s:(s['source_id'],'gemma3_4b_it',arm),
        lambda s:(s['source_id'],'qwen3_4b',arm))
    human_pairs.append({'comparison':f'gemma3_4b_it - qwen3_4b: {arm}','n_determinate_pairs':n_pairs,'pairs_excluded_for_cannot_determine':excluded,'error_rate_difference':round(mean,4) if mean is not None else None,'ci95_low':round(lo,4) if lo is not None else None,'ci95_high':round(hi,4) if hi is not None else None,'annotator_count':1})
with (RUN_DIR/'paired_human_error_bootstrap.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(human_pairs[0]));w.writeheader();w.writerows(human_pairs)
print('Paired human error-rate differences saved; descriptive single-annotator pilot only.')

public_zip=RUN_DIR/'factindic_aggregate_results_no_source_data.zip'
public_metadata={k:v for k,v in run_metadata.items() if k not in {'run_id','input_sha256','selected_source_line_ids'}}
public_metadata['sample_design']=f'{len(sample)} source pairs from official h2e test split; deterministic seed {SEED}'
(RUN_DIR/'public_run_metadata.json').write_text(json.dumps(public_metadata,ensure_ascii=False,indent=2),encoding='utf-8')
public_files=['public_run_metadata.json','quality_audit.json','rouge_summary.csv','paired_rouge_bootstrap.csv','human_factuality_summary.csv','paired_human_error_bootstrap.csv']
with zipfile.ZipFile(public_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in public_files:
        path=RUN_DIR/name
        if path.exists(): z.write(path,arcname=name)
print('Aggregate-only results pack (no sources, references, generations, or row IDs):',public_zip)
files.download(str(public_zip))


## References and reporting limits

- GupShup: [EMNLP 2021 paper](https://aclanthology.org/2021.emnlp-main.499/) and [official release repository](https://github.com/midas-research/gupshup). The source paper describes manually translating SAMSum dialogues into Hi-En; state that limitation.
- [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B) and [Gemma 3 4B IT model card](https://huggingface.co/google/gemma-3-4b-it).
- Google Colab GPU access is quota/availability dependent; see the [official FAQ](https://research.google.com/colaboratory/faq.html).

Do not report ROUGE as factuality. Do not report the automated pilot/NLI results from Devanagari data as this study. For this 30-source pilot, describe the results as exploratory and single-annotator.